In [6]:
import os
import glob
import time
import pandas as pd
from pandas.errors import EmptyDataError
from selenium import webdriver
from selenium.webdriver.common.by import By
from tqdm import tqdm


In [7]:
# CSV 파일들이 있는 폴더 경로(예시)
# lifestyle-entertainment, IoT, Support, 
# AppFamilies, BizIntel, Website-AppBuilding, HumanResources, ArtificialIntelligence, ContentFiles, Productivity, ITOperations, Communication, Commerce, Marketing
# SalesCRM
PATH = r"D:\code\kka_1\data\Zap_Template_data\SalesCRM"

# 폴더 내 모든 csv 파일 탐색
csv_files = glob.glob(os.path.join(PATH, "*.csv"))


In [8]:
print(len(csv_files), type(csv_files))
print(len(csv_files[0]), csv_files[0])


1575 <class 'list'>
72 D:\code\kka_1\data\Zap_Template_data\SalesCRM\10xCRM_ZapTemplateList.csv


In [9]:
test_file_dir, test_file_name = os.path.split(csv_files[0])
test_file_base, test_file_ext = os.path.splitext(test_file_name)
# print(test_file_dir, test_file_base, test_file_ext)
print(test_file_name)
print(csv_files[2])


10xCRM_ZapTemplateList.csv
D:\code\kka_1\data\Zap_Template_data\SalesCRM\17hats_ZapTemplateList.csv


In [11]:
# 크롬 드라이버 초기화
driver = webdriver.Chrome()
driver.maximize_window()
empty_file = []
for file_path in tqdm(csv_files[961:]):
    # # CSV 읽기
    # df = pd.read_csv(file_path, encoding='utf-8', dtype=str)
    try:
        # CSV 읽기
        df = pd.read_csv(file_path, encoding='utf-8', dtype=str)
    except EmptyDataError:
        print(f"EmptyDataError: '{file_path}'는 빈 파일입니다. 건너뜁니다.")
        empty_file.append(file_path)
        continue
    except Exception as e:
        print(f"파일 읽기 오류 발생: '{file_path}' - {e}")
        continue
    # detail 폴더에 저장하기 위해 경로 설정
    file_dir, file_name = os.path.split(file_path)
    file_base, file_ext = os.path.splitext(file_name)
    new_file_name = file_base + "_detail" + file_ext
    # 진행 상황 알기 위해 현재 수집중인 파일 이름 출력
    print("현재 수집 대상 파일: {}".format(file_name))

    # detail 폴더 생성 (이미 존재하면 통과)
    detail_folder = os.path.join(file_dir, "detail")
    os.makedirs(detail_folder, exist_ok=True)

    # URL 칼럼 존재 여부 확인
    if 'URL' not in df.columns:
        print(f"'{file_path}'에서 'URL' 칼럼을 찾지 못했습니다. 건너뜁니다.")
        continue

    # Component 최대 개수를 파악하기 위해 일단 0으로 설정
    max_components = 0
    
    # 수집 데이터를 임시로 저장할 딕셔너리(행 인덱스별로 Component 리스트 보관)
    collected_data = {idx: [] for idx in df.index}
    
    for idx, row in df.iterrows():
        url = row['URL']
        if pd.isna(url):
            # URL이 없거나 NaN이면 그냥 넘어감
            continue        
        try:
            driver.get(url)
            time.sleep(2)  # 페이지 로딩 대기 (필요에 따라 조정)
            # ol/li 요소들 전체를 찾기 위해 최상위 경로만 지정
            li_elements = driver.find_elements(By.XPATH, '//*[@id="main"]/div[1]/main/div[3]/div[1]/section/div[4]/ol/li')            
            # 해당 행에서 찾은 li 개수
            li_count = len(li_elements)
            max_components = max(max_components, li_count)

            # 각 li별로 데이터 추출
            for i in range(li_count):
                # li[i]에 대해서 XPATH 인덱스가 1부터 시작하므로 i+1을 사용
                base_xpath = f'//*[@id="main"]/div[1]/main/div[3]/div[1]/section/div[4]/ol/li[{i+1}]'
                # AppName
                try:
                    app_name = driver.find_element(
                        By.XPATH,
                        base_xpath + '/div[1]/div/div/div/div/div[1]/span[2]/span/img'
                    ).get_attribute('alt')
                except:
                    app_name = ''
                # TnA_Category
                try:
                    tna_category = driver.find_element(
                        By.XPATH,
                        base_xpath + '/div[2]/div/div[1]/span[1]'
                    ).text.strip()
                except:
                    tna_category = ''
                # TnA_Do
                try:
                    tna_do = driver.find_element(
                        By.XPATH,
                        base_xpath + '/div[2]/div/div[1]/span[2]/span/span[2]'
                    ).text.strip()
                except:
                    tna_do = ''
                # TnA_Name
                try:
                    tna_name = driver.find_element(
                        By.XPATH,
                        base_xpath + '/div[1]/div/div/div/div/div[2]/div/strong'
                    ).text.strip()
                except:
                    tna_name = ''
                # TnA_Description
                try:
                    tna_desc = driver.find_element(
                        By.XPATH,
                        base_xpath + '/div[1]/div/div/div/div/div[2]/div/div/p'
                    ).text.strip()
                except:
                    tna_desc = ''
                # 지정된 형식으로 합치기
                component_string = f"{app_name}~{tna_category}~{tna_do}+{tna_name}~{tna_desc}"
                collected_data[idx].append(component_string)
        except Exception as e:
            print(f"에러 발생 (행: {idx}, URL: {url}): {e}")
            continue
    # 데이터프레임에 Component 컬럼 추가
    # 최대 max_components 만큼 컬럼을 만들어두고, 없는 경우는 빈 문자열로 채움
    for i in range(max_components):
        col_name = f"Component{i+1}"
        df[col_name] = ''  # 일단 컬럼이 없으면 생성
    # 실제 수집된 데이터 반영
    for idx in df.index:
        components = collected_data[idx]
        for comp_idx, comp_val in enumerate(components):
            df.at[idx, f"Component{comp_idx+1}"] = comp_val
    # 새 파일 경로
    new_file_path = os.path.join(detail_folder, new_file_name)    
    df.to_csv(new_file_path, index=False, encoding='utf-8-sig')
    print(f"저장 완료: {new_file_path}")
# 크롬 드라이버 종료
driver.quit()

  0%|          | 0/614 [00:00<?, ?it/s]

현재 수집 대상 파일: Outgrow_ZapTemplateList.csv


  0%|          | 1/614 [1:03:01<643:49:34, 3781.04s/it]

저장 완료: D:\code\kka_1\data\Zap_Template_data\SalesCRM\detail\Outgrow_ZapTemplateList_detail.csv
현재 수집 대상 파일: Outplay_ZapTemplateList.csv


  0%|          | 2/614 [1:07:59<294:30:29, 1732.40s/it]

...

저장 완료: D:\code\kka_1\data\Zap_Template_data\SalesCRM\detail\Zoho Survey_ZapTemplateList_detail.csv
현재 수집 대상 파일: Zomentum_ZapTemplateList.csv


100%|█████████▉| 613/614 [81:47:34<25:04, 1504.08s/it]  

저장 완료: D:\code\kka_1\data\Zap_Template_data\SalesCRM\detail\Zomentum_ZapTemplateList_detail.csv
현재 수집 대상 파일: Zonka Feedback_ZapTemplateList.csv


100%|██████████| 614/614 [81:49:15<00:00, 479.73s/it] 


저장 완료: D:\code\kka_1\data\Zap_Template_data\SalesCRM\detail\Zonka Feedback_ZapTemplateList_detail.csv
